In [1]:
from transformers import AutoModelForMaskedLM , AutoTokenizer
import torch
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import re
from tqdm.auto import tqdm

In [2]:
class Prompting(object):
  """ doc string 
   This class helps us to implement
   Prompt-based Learning Model
  """
  def __init__(self, **kwargs):
    """ constructor 

    parameter:
    ----------
       model: AutoModelForMaskedLM
            path to a Pre-trained language model form HuggingFace Hub
       tokenizer: AutoTokenizer
            path to tokenizer if different tokenizer is used, 
            otherwise leave it empty
    """
    model_path=kwargs['model']
    tokenizer_path= kwargs['model']
    if "tokenizer" in kwargs.keys():
      tokenizer_path= kwargs['tokenizer']
    self.model = AutoModelForMaskedLM.from_pretrained(model_path)
    self.tokenizer = AutoTokenizer.from_pretrained(model_path)

  def prompt_pred(self,text):
    """
      Predict MASK token by listing the probability of candidate tokens 
      where the first token is the most likely

      Parameters:
      ----------
      text: str 
          The text including [MASK] token.
          It supports single MASK token. If more [MASK] tokens 
          are given, it takes the first one.

      Returns:
      --------
      list of (token, prob)
         The return is a list of all token in LM Vocab along with 
         their prob score, sort by score in descending order 
    """
    indexed_tokens=self.tokenizer(text, return_tensors="pt").input_ids
    tokenized_text= self.tokenizer.convert_ids_to_tokens (indexed_tokens[0])
    # take the first masked token
    mask_pos=tokenized_text.index(self.tokenizer.mask_token)
    self.model.eval()
    with torch.no_grad():
      outputs = self.model(indexed_tokens)
      predictions = outputs[0]
    values, indices=torch.sort(predictions[0, mask_pos],  descending=True)
    #values=torch.nn.functional.softmax(values, dim=0)
    result=list(zip(self.tokenizer.convert_ids_to_tokens(indices), values))
    self.scores_dict={a:b for a,b in result}
    return result

  def compute_tokens_prob(self, text, token_list1, token_list2):
    """
    Compute the activations for given two token list, 

    Parameters:
    ---------
    token_list1: List(str)
     it is a list for positive polarity tokens such as good, great. 
    token_list2: List(str)
     it is a list for negative polarity tokens such as bad, terrible.      

    Returns:
    --------
    Tuple (
       the probability for first token list,
       the probability of the second token list,
       the ratio score1/ (score1+score2)
       The softmax returns
    """
    _=self.prompt_pred(text)
    score1=[self.scores_dict[token1] if token1 in self.scores_dict.keys() else 0\
            for token1 in token_list1]
    score1= sum(score1)
    score2=[self.scores_dict[token2] if token2 in self.scores_dict.keys() else 0\
            for token2 in token_list2]
    score2= sum(score2)
    softmax_rt=torch.nn.functional.softmax(torch.Tensor([score1,score2]), dim=0)
    return softmax_rt

prompting= Prompting(model='bert-base-uncased')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
import pandas as pd
df = pd.read_csv('/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
def clean_text(text):
    # Lowercase (for uncased models)
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove URLs
    text = re.sub(r'(https?://\S+|www\.\S+)', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['review'] = df['review'].apply(clean_text)


le = LabelEncoder()
df['sentiment'] = le.fit_transform(df['sentiment'])
prompt = "Because it was [MASK]."

def predict_sentiment(text):
    # lấy 200 từ cuối
    text = " ".join(text.split()[-200:])
    
    res = prompting.compute_tokens_prob(
        text + prompt,
        token_list1=["good", "amazing", "fantastic"],
        token_list2=["bad", "awful", "terrible"]
    )
    
    # res[0] = prob positive
    return 1 if res[0] > 0.8 else 0

subset_df = df.copy()

preds = []
for review in tqdm(subset_df['review'], desc="Processing reviews"):
    preds.append(predict_sentiment(review))

subset_df['pred'] = preds

In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

y_true = subset_df['sentiment']
y_pred = subset_df['pred']

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1-score:", f1_score(y_true, y_pred))

print(classification_report(y_true, y_pred))

Accuracy: 0.76192
Precision: 0.7442007906317595
Recall: 0.7982
F1-score: 0.7702551433975373
              precision    recall  f1-score   support

           0       0.78      0.73      0.75     25000
           1       0.74      0.80      0.77     25000

    accuracy                           0.76     50000
   macro avg       0.76      0.76      0.76     50000
weighted avg       0.76      0.76      0.76     50000



In [12]:
# load Prompting class

prompt=" Because it was [MASK]."
text="I really like the film a lot."
prompting.prompt_pred(text+prompt)[:10]

[('great', tensor(9.5558)),
 ('amazing', tensor(9.2532)),
 ('good', tensor(9.1464)),
 ('fun', tensor(8.3979)),
 ('fantastic', tensor(8.3277)),
 ('wonderful', tensor(8.2719)),
 ('beautiful', tensor(8.1584)),
 ('awesome', tensor(8.1071)),
 ('incredible', tensor(8.0140)),
 ('funny', tensor(7.8785))]

In [10]:
prompting.compute_tokens_prob("This movie is not worth to watch. Because it was [MASK].", token_list1=["good", "amazing", "fantastic"], token_list2= ["bad", "awful", "terrible"])

tensor([0.3192, 0.6808])

In [3]:
import pandas as pd
subset_df = pd.read_csv('/kaggle/input/datasets/nguynnamhong1/manual-cloze-prompt/predicted_imdb.csv')

In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

y_true = subset_df['sentiment']
y_pred = subset_df['pred']

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1-score:", f1_score(y_true, y_pred))

print(classification_report(y_true, y_pred))

Accuracy: 0.76192
Precision: 0.7442007906317595
Recall: 0.7982
F1-score: 0.7702551433975373
              precision    recall  f1-score   support

           0       0.78      0.73      0.75     25000
           1       0.74      0.80      0.77     25000

    accuracy                           0.76     50000
   macro avg       0.76      0.76      0.76     50000
weighted avg       0.76      0.76      0.76     50000

